In [1]:
from pathlib import Path
import math
import warnings
from itertools import product

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from scipy.stats import weibull_min
from shapely.geometry import Point, Polygon


from py_wake.site import XRSite
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine
from py_wake.deficit_models import TurboGaussianDeficit
from py_wake.superposition_models import SquaredSum
from py_wake.wind_farm_models import All2AllIterative

DATA_DIR = Path('../data')
ERA5_DIR = DATA_DIR / 'era5'
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

/rds/general/user/vk825/home/envs/wake_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1. Load farm polygons

In [2]:
candidate_farms = gpd.read_file(DATA_DIR / 'candidate_farms.gpkg')
neighbour_farms = gpd.read_file(DATA_DIR / 'neighbor_farms.gpkg')

print(type(candidate_farms))
print(candidate_farms.head())
print(neighbour_farms.head())

<class 'geopandas.geodataframe.GeoDataFrame'>
                name  country      status  power_mw  n_turbines  area_sqkm  \
0            Norther  Belgium  Production     369.6        44.0  38.442050   
1            C-Power  Belgium  Production     325.2        54.0  19.852823   
2             Rentel  Belgium  Production     309.0        42.0  23.265816   
3  Seamade (SeaStar)  Belgium  Production     252.0        30.0  18.425859   
4            Mermaid  Belgium  Production     235.2        28.0  16.267913   

  type_inst                                           geometry  
0  Grounded  MULTIPOLYGON (((502777.3 5706764.815, 498146.6...  
1  Grounded  MULTIPOLYGON (((495673.008 5707997.117, 495000...  
2  Grounded  MULTIPOLYGON (((492188.73 5715538.777, 493455....  
3  Grounded  MULTIPOLYGON (((490574.205 5718251.336, 489176...  
4       NaN  MULTIPOLYGON (((483097 5728870, 480483 5726839...  
                name      country      status  power_mw  n_turbines  \
0   Borssele Kavel V  Ne

In [4]:
# Select target farm
target_name = 'Norther'
target_gdf = candidate_farms.loc[candidate_farms['name'] == target_name].copy()

if target_gdf.empty:
    raise ValueError(f'Could not find target farm: {target_name}')

if target_gdf.crs.is_geographic:
    target_gdf = target_gdf.to_crs(epsg=32631)
    neighbour_farms = neighbour_farms.to_crs(epsg=32631)
else:
    neighbour_farms = neighbour_farms.to_crs(target_gdf.crs)

target_poly = target_gdf.geometry.iloc[0]
target_centroid = target_poly.centroid

print(target_gdf[['name', 'power_mw', 'n_turbines', 'geometry']])
print('CRS:', target_gdf.crs)
print('Target centroid:', target_centroid.x, target_centroid.y)

      name  power_mw  n_turbines  \
0  Norther     369.6        44.0   

                                            geometry  
0  MULTIPOLYGON (((502777.3 5706764.815, 498146.6...  
CRS: EPSG:32631
Target centroid: 500911.69264268246 5708425.632954922


#### 2. Load ERA5 and calculate wind speed/direction

In [5]:
ds = xr.open_mfdataset(str(ERA5_DIR / 'era5_norther_100m_*.nc'))

time_name = 'valid_time' if 'valid_time' in ds.coords else 'time'

print(f'Loaded {ds.sizes[time_name]} records')
print(f'Period: {str(ds[time_name].values[0])[:10]} to {str(ds[time_name].values[-1])[:10]}')
print(f'Variables: {list(ds.data_vars)}')


Loaded 26304 records
Period: 2023-01-01 to 2025-12-31
Variables: ['u100', 'v100']


In [6]:
# Average across ERA5 grid points around the target site
spatial_dims = [d for d in ['latitude', 'longitude'] if d in ds['u100'].dims]
u100 = ds['u100'].mean(dim=spatial_dims).values
v100 = ds['v100'].mean(dim=spatial_dims).values

# Wind speed
ws_era5 = np.sqrt(u100**2 + v100**2)

# Meteorological wind direction: direction wind is COMING FROM.
# 0 deg = from North, 90 deg = from East.
wd_era5 = (270 - np.degrees(np.arctan2(v100, u100))) % 360

print(f'Mean wind speed: {ws_era5.mean():.2f} m/s')
print(f'Min/Max wind speed: {ws_era5.min():.2f} / {ws_era5.max():.2f} m/s')

Mean wind speed: 8.80 m/s
Min/Max wind speed: 0.00 / 27.65 m/s


#### 3. Build 12-sector Weibull wind rose for AEP calculations


In [7]:
wd_centres = np.arange(0, 360, 30)
wind_rose_rows = []

for wd in wd_centres:
    lower = (wd - 15) % 360
    upper = (wd + 15) % 360

    if lower > upper:
        mask = (wd_era5 >= lower) | (wd_era5 < upper)
    else:
        mask = (wd_era5 >= lower) & (wd_era5 < upper)

    speeds = ws_era5[mask]
    speeds = speeds[np.isfinite(speeds) & (speeds > 0)]

    frequency = len(speeds) / len(ws_era5)

    if len(speeds) < 5:
        A, k = np.nan, np.nan
    else:
        k, loc, A = weibull_min.fit(speeds, floc=0)

    wind_rose_rows.append({
        'wd': wd,
        'sector_frequency': frequency,
        'weibull_A': A,
        'weibull_k': k,
        'n_samples': len(speeds),
    })

wind_rose = pd.DataFrame(wind_rose_rows)
print(wind_rose)
print('Frequency sum:', wind_rose['sector_frequency'].sum())


     wd  sector_frequency  weibull_A  weibull_k  n_samples
0     0          0.056037   8.444577   2.267462       1474
1    30          0.081851   8.858454   2.784762       2153
2    60          0.071244   8.853347   2.597216       1874
3    90          0.056037   8.141152   2.515084       1474
4   120          0.053224   7.208759   2.369640       1400
5   150          0.048054   7.768672   2.140386       1264
6   180          0.090062  10.526288   2.297022       2369
7   210          0.128574  11.510080   2.392055       3382
8   240          0.197004  11.895142   2.707229       5182
9   270          0.099034  10.311149   2.142304       2605
10  300          0.065845   9.489744   2.042344       1732
11  330          0.053034   8.522368   2.015897       1395
Frequency sum: 1.0


In [8]:
sector_freq = wind_rose['sector_frequency'].to_numpy()
weibull_A = wind_rose['weibull_A'].to_numpy()
weibull_k = wind_rose['weibull_k'].to_numpy()
wd_centres = wind_rose['wd'].to_numpy()

def create_site(ti=0.06):
    """Create PyWake site with wind rose and turbulence intensity."""
    return XRSite(
        ds=xr.Dataset(
            data_vars={
                'Sector_frequency': ('wd', sector_freq),
                'Weibull_A': ('wd', weibull_A),
                'Weibull_k': ('wd', weibull_k),
                'TI': ti,
            },
            coords={'wd': wd_centres}
        )
    )

In [9]:
def create_turbine(spec):
    """Create a generic wind turbine. power is in kW for PyWake GenericWindTurbine."""
    return GenericWindTurbine(
        name=spec['name'],
        diameter=spec['diameter'],
        hub_height=spec['hub_height'],
        power_norm=spec['power_kw'],
        turbulence_intensity=spec.get('turbulence_intensity', 0.06),
    )

In [10]:
def create_wake_model(turbine_spec, ti=0.06):
    site = create_site(ti=ti)
    wt = create_turbine(turbine_spec)
    return All2AllIterative(
        site, wt,
        wake_deficitModel=TurboGaussianDeficit(),
        superpositionModel=SquaredSum(),
    )

In [11]:
def place_turbines_in_polygon(
    polygon,
    n_turbines,
    min_spacing_D=5,
    rotor_diameter=164,
    seed=42,
):
    """
    Place turbines inside a polygon on a regular grid.

    This is a simple plausible-layout generator. It is deterministic for a given seed.
    If the polygon cannot fit the requested number of turbines at the requested spacing,
    it returns the maximum number it can place and prints a warning.
    """
    if n_turbines <= 0:
        return np.array([]), np.array([])

    local_rng = np.random.default_rng(seed)
    min_spacing_m = min_spacing_D * rotor_diameter
    minx, miny, maxx, maxy = polygon.bounds

    # Estimated spacing tries to spread turbines across the whole polygon.
    estimated_spacing = np.sqrt(polygon.area / n_turbines) * 0.9
    spacing = max(estimated_spacing, min_spacing_m)

    x_values = np.arange(minx + spacing / 2, maxx, spacing)
    y_values = np.arange(miny + spacing / 2, maxy, spacing)

    candidates = []
    for x in x_values:
        for y in y_values:
            p = Point(x, y)
            if polygon.contains(p):
                candidates.append((x, y))

    if len(candidates) == 0:
        return np.array([]), np.array([])

    if len(candidates) < n_turbines:
        selected = np.array(candidates)
    else:
        idx = local_rng.choice(len(candidates), size=n_turbines, replace=False)
        selected = np.array(candidates)[idx]

    return selected[:, 0], selected[:, 1]

def create_synthetic_neighbour(target_centroid, distance_km, direction_deg, width_km=4, depth_km=4):
    """Create a rectangular synthetic neighbour polygon around the target farm.

    direction_deg is bearing from the target to the neighbour.
    0 = North, 90 = East.
    """
    dist_m = distance_km * 1000
    theta = np.deg2rad(direction_deg)

    center_x = target_centroid.x + dist_m * np.sin(theta)
    center_y = target_centroid.y + dist_m * np.cos(theta)

    half_w = width_km * 1000 / 2
    half_d = depth_km * 1000 / 2

    return Polygon([
        (center_x - half_w, center_y - half_d),
        (center_x + half_w, center_y - half_d),
        (center_x + half_w, center_y + half_d),
        (center_x - half_w, center_y + half_d),
    ])


In [12]:
def bearing_from_dxdy(dx, dy):
    """Bearing in degrees from dx/dy where 0=N, 90=E."""
    return (np.degrees(np.arctan2(dx, dy)) + 360) % 360

def flow_unit_vector_from_wd(wd_deg):
    """
    Convert meteorological wind direction to flow unit vector.
    wd_deg = direction wind comes FROM.
    Returned vector points where wind is going TO.
    """
    to_dir = (wd_deg + 180) % 360
    theta = np.deg2rad(to_dir)
    return np.sin(theta), np.cos(theta)

def neighbour_static_features(target_poly, nb_poly, nb_turbines, nb_spacing_D, nb_turbine_spec):
    """Features that do not depend on wind direction."""
    tc = target_poly.centroid
    nc = nb_poly.centroid
    dx = nc.x - tc.x
    dy = nc.y - tc.y
    distance_km = np.sqrt(dx**2 + dy**2) / 1000
    bearing_deg = bearing_from_dxdy(dx, dy)
    area_sqkm = nb_poly.area / 1e6
    capacity_mw = nb_turbines * nb_turbine_spec['power_kw'] / 1000
    density_turbines_per_sqkm = nb_turbines / area_sqkm if area_sqkm > 0 else np.nan
    capacity_density_mw_per_sqkm = capacity_mw / area_sqkm if area_sqkm > 0 else np.nan

    return {
        'nb_distance_km': distance_km,
        'nb_direction_deg': bearing_deg,
        'nb_area_sqkm': area_sqkm,
        'nb_n_turbines': nb_turbines,
        'nb_spacing_D': nb_spacing_D,
        'nb_capacity_mw': capacity_mw,
        'nb_density_turbines_per_sqkm': density_turbines_per_sqkm,
        'nb_capacity_density_mw_per_sqkm': capacity_density_mw_per_sqkm,
    }

def wind_relative_features(target_poly, nb_poly, wd_deg):
    """
    Wind-relative topology features.

    streamwise_distance_km is positive when the neighbour is upwind of the target.
    lateral_distance_km is the side offset from the wind-flow line.
    """
    tc = target_poly.centroid
    nc = nb_poly.centroid
    dx = nc.x - tc.x
    dy = nc.y - tc.y

    fx, fy = flow_unit_vector_from_wd(wd_deg)

    # Vector target -> neighbour dotted with flow vector.
    # Negative means neighbour is upstream of target, so multiply by -1.
    streamwise_m = -(dx * fx + dy * fy)

    # 2D cross product magnitude gives lateral offset.
    lateral_m_signed = dx * fy - dy * fx

    return {
        'streamwise_distance_km': streamwise_m / 1000,
        'lateral_distance_km': lateral_m_signed / 1000,
        'abs_lateral_distance_km': abs(lateral_m_signed) / 1000,
        'is_upwind_neighbour': int(streamwise_m > 0),
    }

def add_cyclic_wind_features(row):
    wd_rad = np.deg2rad(row['wind_direction'])
    row['wd_sin'] = np.sin(wd_rad)
    row['wd_cos'] = np.cos(wd_rad)
    return row


In [13]:
# Norther: 44 x Vestas V164-8.4 MW. Use fixed target spec.
target_turbine_spec = {
    'name': 'Vestas_V164_8p4MW',
    'diameter': 164,
    'hub_height': 105,
    'power_kw': 8400,
    'turbulence_intensity': 0.06,
}

# For future neighbours you can test larger turbines. Start simple.
neighbour_turbine_specs = [
    {'name': 'Vestas_V164_8p4MW', 'diameter': 164, 'hub_height': 105, 'power_kw': 8400, 'turbulence_intensity': 0.06},
    {'name': 'Generic_10MW', 'diameter': 178, 'hub_height': 119, 'power_kw': 10000, 'turbulence_intensity': 0.06},
    {'name': 'Generic_15MW', 'diameter': 240, 'hub_height': 150, 'power_kw': 15000, 'turbulence_intensity': 0.06},
]

target_n_turbines = 44
target_spacing_D = 5

x_target, y_target = place_turbines_in_polygon(
    target_poly,
    n_turbines=target_n_turbines,
    min_spacing_D=target_spacing_D,
    rotor_diameter=target_turbine_spec['diameter'],
    seed=RANDOM_SEED,
)

print('Target turbines placed:', len(x_target))
if len(x_target) != target_n_turbines:
    print('Warning: target layout has fewer turbines than expected. Check polygon area/CRS/spacing.')


Target turbines placed: 44


In [14]:
def power_for_condition(wake_model, x, y, wd, ws):
    """Return total instantaneous power for all turbines for one wind condition."""
    sim = wake_model(x, y, wd=[wd], ws=[ws])
    return float(sim.Power.squeeze().sum().values)

def target_power_for_condition(wake_model, all_x, all_y, n_target, wd, ws):
    """Return instantaneous power for target turbines only."""
    sim = wake_model(all_x, all_y, wd=[wd], ws=[ws])
    return float(sim.Power.isel(wt=slice(0, n_target)).squeeze().sum().values)

def simulate_condition_level_scenario(
    scenario_id,
    nb_poly,
    nb_n_turbines,
    nb_spacing_D,
    nb_turbine_spec,
    ti,
    wind_speeds,
    wind_directions,
    layout_seed,
):
    """
    Simulate one neighbour scenario for many wind speed/direction pairs.
    Returns one row per wind condition.
    """
    # For now, use one turbine type for all turbines in the PyWake model.
    # If you need mixed target/neighbor turbine types later, use WindTurbines with multiple types.
    turbine_spec = nb_turbine_spec
    wake_model = create_wake_model(turbine_spec, ti=ti)

    x_nb, y_nb = place_turbines_in_polygon(
        nb_poly,
        n_turbines=nb_n_turbines,
        min_spacing_D=nb_spacing_D,
        rotor_diameter=turbine_spec['diameter'],
        seed=layout_seed,
    )

    if len(x_nb) == 0:
        return []

    all_x = np.concatenate([x_target, x_nb])
    all_y = np.concatenate([y_target, y_nb])
    n_target = len(x_target)

    static_feats = neighbour_static_features(
        target_poly, nb_poly, len(x_nb), nb_spacing_D, turbine_spec
    )

    rows = []
    for wd, ws in product(wind_directions, wind_speeds):
        p_alone = power_for_condition(wake_model, x_target, y_target, wd=wd, ws=ws)
        p_with = target_power_for_condition(wake_model, all_x, all_y, n_target=n_target, wd=wd, ws=ws)

        if p_alone > 0:
            wake_loss_pct = (1 - p_with / p_alone) * 100
        else:
            wake_loss_pct = np.nan

        wind_feats = wind_relative_features(target_poly, nb_poly, wd)
        wd_rad = np.deg2rad(wd)

        rows.append({
            'scenario_id': scenario_id,
            'wind_speed': ws,
            'wind_direction': wd,
            'wd_sin': np.sin(wd_rad),
            'wd_cos': np.cos(wd_rad),
            'ti': ti,
            'turbine_name': turbine_spec['name'],
            'rotor_diam_m': turbine_spec['diameter'],
            'hub_height_m': turbine_spec['hub_height'],
            'rated_power_kw': turbine_spec['power_kw'],
            'target_n_turbines': n_target,
            'target_spacing_D': target_spacing_D,
            **static_feats,
            **wind_feats,
            'power_alone_w': p_alone,
            'power_with_neighbour_w': p_with,
            'wake_loss_pct': wake_loss_pct,
        })

    return rows

def simulate_aep_scenario(
    scenario_id,
    nb_poly,
    nb_n_turbines,
    nb_spacing_D,
    nb_turbine_spec,
    ti,
    layout_seed,
):
    """
    AEP-level scenario using ERA5-derived Weibull wind rose.
    This produces one row per scenario.
    """
    turbine_spec = nb_turbine_spec
    wake_model = create_wake_model(turbine_spec, ti=ti)

    x_nb, y_nb = place_turbines_in_polygon(
        nb_poly,
        n_turbines=nb_n_turbines,
        min_spacing_D=nb_spacing_D,
        rotor_diameter=turbine_spec['diameter'],
        seed=layout_seed,
    )

    if len(x_nb) == 0:
        return None

    all_x = np.concatenate([x_target, x_nb])
    all_y = np.concatenate([y_target, y_nb])
    n_target = len(x_target)

    sim_alone = wake_model(x_target, y_target, wd=wd_centres, ws=np.arange(3, 26, 1))
    aep_alone = float(sim_alone.aep().sum().values)

    sim_with = wake_model(all_x, all_y, wd=wd_centres, ws=np.arange(3, 26, 1))
    aep_with = float(sim_with.aep().isel(wt=slice(0, n_target)).sum().values)

    wake_loss_pct = (1 - aep_with / aep_alone) * 100 if aep_alone > 0 else np.nan

    static_feats = neighbour_static_features(target_poly, nb_poly, len(x_nb), nb_spacing_D, turbine_spec)

    return {
        'scenario_id': scenario_id,
        'ti': ti,
        'turbine_name': turbine_spec['name'],
        'rotor_diam_m': turbine_spec['diameter'],
        'hub_height_m': turbine_spec['hub_height'],
        'rated_power_kw': turbine_spec['power_kw'],
        'target_n_turbines': n_target,
        'target_spacing_D': target_spacing_D,
        **static_feats,
        'aep_alone_gwh': aep_alone,
        'aep_with_neighbour_gwh': aep_with,
        'wake_loss_pct': wake_loss_pct,
    }


In [16]:
# Keep this moderate first. Increase later on the HPC.
nb_distances_km = [5, 10, 15, 20, 25, 30]
nb_bearings_deg = np.arange(0, 360, 30)
nb_sizes = [20, 40, 60, 80]
nb_spacings_D = [5, 7, 9]
ti_values = [0.04, 0.06, 0.08, 0.10]

synthetic_scenarios = []
scenario_id = 0

for dist_km, bearing_deg, nb_n, nb_spacing, ti, nb_spec in product(
    nb_distances_km, nb_bearings_deg, nb_sizes, nb_spacings_D, ti_values, neighbour_turbine_specs
):
    scenario_id += 1
    nb_poly = create_synthetic_neighbour(
        target_centroid,
        distance_km=dist_km,
        direction_deg=bearing_deg,
        width_km=5,
        depth_km=5,
    )
    synthetic_scenarios.append({
        'scenario_id': scenario_id,
        'nb_poly': nb_poly,
        'nb_n_turbines': nb_n,
        'nb_spacing_D': nb_spacing,
        'nb_turbine_spec': nb_spec,
        'ti': ti,
        'layout_seed': RANDOM_SEED + scenario_id,
    })

print('Synthetic scenarios:', len(synthetic_scenarios))


Synthetic scenarios: 10368


In [17]:
# Wind condition grid for training.
wind_speeds = np.arange(3, 26, 1)
wind_directions = np.arange(0, 360, 20)

# Quick test first. Change to synthetic_scenarios for full run.
SCENARIOS_TO_RUN = synthetic_scenarios.copy()

all_rows = []
for i, s in enumerate(SCENARIOS_TO_RUN, start=1):
    print(f'Running scenario {i}/{len(SCENARIOS_TO_RUN)}: id={s["scenario_id"]}')
    try:
        rows = simulate_condition_level_scenario(
            scenario_id=s['scenario_id'],
            nb_poly=s['nb_poly'],
            nb_n_turbines=s['nb_n_turbines'],
            nb_spacing_D=s['nb_spacing_D'],
            nb_turbine_spec=s['nb_turbine_spec'],
            ti=s['ti'],
            wind_speeds=wind_speeds,
            wind_directions=wind_directions,
            layout_seed=s['layout_seed'],
        )
        all_rows.extend(rows)
    except Exception as e:
        print(f'  Failed scenario {s["scenario_id"]}: {e}')

    if i % 10 == 0:
        pd.DataFrame(all_rows).to_csv(OUTPUT_DIR / 'training_data_condition_level_partial.csv', index=False)

df_condition = pd.DataFrame(all_rows)
condition_path = OUTPUT_DIR / 'training_data_condition_level.csv'
df_condition.to_csv(condition_path, index=False)

print('Rows:', len(df_condition))
print('Saved:', condition_path)
df_condition.head()


Running scenario 1/5: id=1
Running scenario 2/5: id=2
Running scenario 3/5: id=3


/var/tmp/pbs.3018876.pbs-7/ipykernel_4061721/1249646187.py:41: UserWarning: Only 16 turbines fit inside polygon, requested 20. Returning 16 turbines.
  warnings.warn(


Running scenario 4/5: id=4
Running scenario 5/5: id=5
Rows: 2070
Saved: ../data/training_data_condition_level.csv


,scenario_id,wind_speed,wind_direction,wd_sin,wd_cos,ti,turbine_name,rotor_diam_m,hub_height_m,rated_power_kw,...,nb_capacity_mw,nb_density_turbines_per_sqkm,nb_capacity_density_mw_per_sqkm,streamwise_distance_km,lateral_distance_km,abs_lateral_distance_km,is_upwind_neighbour,power_alone_w,power_with_neighbour_w,wake_loss_pct
0,1,3,0,0.0,1.0,0.04,Vestas_V164_8p4MW,164,105,8400,...,168.0,0.8,6.72,5.0,-5.881998e-14,5.881998e-14,1,8.963401e+05,7.673220e+05,14.393878
1,1,4,0,0.0,1.0,0.04,Vestas_V164_8p4MW,164,105,8400,...,168.0,0.8,6.72,5.0,-5.881998e-14,5.881998e-14,1,3.336057e+06,2.924920e+06,12.324017
2,1,5,0,0.0,1.0,0.04,Vestas_V164_8p4MW,164,105,8400,...,168.0,0.8,6.72,5.0,-5.881998e-14,5.881998e-14,1,7.815591e+06,6.902262e+06,11.685980
3,1,6,0,0.0,1.0,0.04,Vestas_V164_8p4MW,164,105,8400,...,168.0,0.8,6.72,5.0,-5.881998e-14,5.881998e-14,1,1.489925e+07,1.320880e+07,11.345853
4,1,7,0,0.0,1.0,0.04,Vestas_V164_8p4MW,164,105,8400,...,168.0,0.8,6.72,5.0,-5.881998e-14,5.881998e-14,1,2.510601e+07,2.229354e+07,11.202401
